# 稀疏数据表示

学习目标：为重复值较多的数据选择稀疏表示，区分填充值与缺失值，检查运算、稠密化和内存估计的适用条件。

前置知识：dtype、缺失值、内存估计、数组转换。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例输入在单元内构造，后续单元沿用首次导入的 pd 和 np；明确说明复用的对象。SciPy 仅介绍互操作入口，不作为本章运行依赖。

## 1 保存大量重复的零

六个门店的退货件数为 0、0、3、0、0、1。零表示没有退货，是有效观测；用稀疏表示可以省略反复出现的零，同时保留六条记录和各自标签。

SparseDtype 指定数据类型和填充值。下面将普通 int64 序列转为以 0 为填充值的稀疏序列，再直接统计退货总量。

In [1]:
import numpy as np
import pandas as pd

dense_returns = pd.Series(
    [0, 0, 3, 0, 0, 1], index=["A", "B", "C", "D", "E", "F"], name="returns", dtype="int64"
)
returns = dense_returns.astype(pd.SparseDtype("int64", fill_value=0))

print(returns)  # 六个门店都保留；C 为 3 件，F 为 1 件，其余为 0。
print(returns.dtype)  # Sparse[int64, 0]。
print(returns.sum(), returns.count())  # 4 件，6 条非缺失观测；零仍参与计数。
print(returns.shape, returns.index.equals(dense_returns.index))  # (6,) True。

A    0
B    0
C    3
D    0
E    0
F    1
Name: returns, dtype: Sparse[int64, 0]
Sparse[int64, 0]
4 6
(6,) True


## 2 SparseArray 与稀疏访问器

SparseArray 是一维扩展数组，保存需要显式存储的值及其位置；Series 在它外面再加行标签和名称。直接从普通数据构造时，等于 fill_value 的位置可以省略。SparseDtype 同时记录底层数值类型 subtype 和填充值。

Series.sparse 提供专门的观察入口。npoints 是实际存储的条目数，density 是这个数量除以序列长度，sp_values 是存储的值数组；它们不等于完整序列。下面重新构造同样的六个数。

In [2]:
array = pd.arrays.SparseArray([0, 0, 3, 0, 0, 1], dtype="int64", fill_value=0)
series = pd.Series(array, index=["A", "B", "C", "D", "E", "F"])

print(array)  # 长度为 6；显式存储位置为 2、5，值为 3、1。
print(array.dtype.subtype, array.dtype.fill_value)  # int64 0。
print(series.sparse.npoints, series.sparse.density)  # 2，约 0.3333，即 2/6。
print(series.sparse.sp_values)  # [3 1]，不是六条完整观测。
print(series.sparse.fill_value)  # 0，未显式存储的位置读取为这个值。

[0, 0, 3, 0, 0, 1]
Fill: 0
IntIndex
Indices: array([2, 5], dtype=int32)

int64 0
2 0.3333333333333333
[3 1]
0


填充值可以显式指定，也有随 dtype 变化的默认值。例如整数默认是 0，浮点数默认是 NaN，布尔值默认是 False。需要省略大量浮点零时，应明确选择 0，而不是假定默认填充值就是零。

In [3]:
print(pd.SparseDtype("int64"))  # Sparse[int64, 0]。
print(pd.SparseDtype("float64"))  # Sparse[float64, nan]。
print(pd.SparseDtype("bool"))  # Sparse[bool, False]。

float_zeros = pd.Series([0.0, 0.0, 2.0], dtype=pd.SparseDtype("float64", 0.0))
print(float_zeros.sparse.npoints)  # 1，只需存储 2.0。
print(float_zeros.dtype)  # Sparse[float64, 0.0]。

Sparse[int64, 0]
Sparse[float64, nan]
Sparse[bool, False]
1
Sparse[float64, 0.0]


## 3 填充值与缺失值

fill_value 决定哪些值可以不单独存储，不是填补缺失值的命令。以 0 为填充值时，输入中的 NaN 仍然缺失，而且需要显式存储；以 NaN 为填充值时，被省略的位置才是缺失。

下面用相同的 float64 输入分别选择 0 和 NaN。isna 检查缺失，count 统计非缺失观测，sum 默认跳过缺失；有效零应参与计数和均值的分母。

In [4]:
values = [0.0, np.nan, 0.0, 4.0]
zero_fill = pd.Series(values, dtype=pd.SparseDtype("float64", 0.0))
nan_fill = pd.Series(values, dtype=pd.SparseDtype("float64", np.nan))

print(zero_fill.sparse.sp_values)  # [nan 4.]：缺失并没有被 0 替换。
print(nan_fill.sparse.sp_values)  # [0. 0. 4.]：两个有效零都需要存储。
print(zero_fill.sparse.density, nan_fill.sparse.density)  # 0.5 0.75，缺失率却相同。
print(zero_fill.isna().to_numpy())  # [False True False False]。
print(zero_fill.count(), zero_fill.sum(), zero_fill.mean())  # 3，4.0，约 1.3333。
print(nan_fill.count(), nan_fill.sum())  # 3，4.0，逻辑数据相同。

[nan  4.]
[0. 0. 4.]
0.5 0.75
[False  True False False]
3 4.0 1.3333333333333333
3 4.0


如果业务规则确定要把缺失改为零，才调用 fillna。它会改变数据含义，而选择稀疏存储的填充值不应改变输入的逻辑值。

下面继续使用 zero_fill。sparse.to_dense 返回普通 Series，保留完整值和标签，便于在小输入上核对。

In [5]:
filled = zero_fill.fillna(0.0)
print(zero_fill.sparse.to_dense().tolist())  # [0.0, nan, 0.0, 4.0]，原序列未改变。
print(filled.sparse.to_dense().tolist())  # [0.0, 0.0, 0.0, 4.0]。
print(filled.count())  # 4；这次确实把缺失变成有效零。
print(zero_fill.sparse.to_dense().equals(nan_fill.sparse.to_dense()))  # True。

[0.0, nan, 0.0, 4.0]
[0.0, 0.0, 0.0, 4.0]
4
True


## 4 非零填充值

稀疏数据不一定是“大部分为零”。如果正常状态用 1 表示，而少数记录为 2，以 1 为填充值可以减少显式条目。填充值改变的是存储方式，不是把全部数据减去 1。

下面从同一份普通输入分别构造两种表示，而不是直接修改已有稀疏数组的填充值。

In [6]:
values = [1, 1, 1, 2]
with_zero = pd.Series(values, dtype=pd.SparseDtype("int64", 0))
with_one = pd.Series(values, dtype=pd.SparseDtype("int64", 1))

print(with_zero.sparse.npoints, with_one.sparse.npoints)  # 4 1。
print(with_zero.sparse.density, with_one.sparse.density)  # 1.0 0.25。
print(with_one.sparse.to_dense().tolist())  # [1, 1, 1, 2]。
print(with_zero.sparse.to_dense().equals(with_one.sparse.to_dense()))  # True。

4 1
1.0 0.25
[1, 1, 1, 2]
True


## 5 多列稀疏表格

DataFrame 的每一列可以使用 SparseDtype；对普通表调用 astype 可以逐列转换。DataFrame.sparse 提供整表密度和稠密化等操作。

下面是四个门店的退货与投诉次数。两列各只有一个非零值，整表共有八个位置，只有两个位置需要显式存储。

In [7]:
dense_events = pd.DataFrame(
    {"returns": [0, 3, 0, 0], "complaints": [0, 0, 2, 0]}, index=["A", "B", "C", "D"]
)
events = dense_events.astype(pd.SparseDtype("int64", 0))

print(events)  # 四行两列；B 的退货为 3，C 的投诉为 2。
print(events.dtypes)  # 两列均为 Sparse[int64, 0]。
print(events.sparse.density)  # 0.25，即 2/8。
print(events.sum())  # 按列求和：returns 为 3，complaints 为 2。
print(events.sparse.to_dense().equals(dense_events))  # True，值、标签与顺序一致。

   returns  complaints
A        0           0
B        3           0
C        0           2
D        0           0
returns       Sparse[int64, 0]
complaints    Sparse[int64, 0]
dtype: object
0.25
returns       3
complaints    2
dtype: Sparse[int64, 0]
True


稀疏列可以与普通列放在同一表中，但整表 sparse 访问器要求各列都使用稀疏 dtype。文本标识不必为了使用访问器而强行稀疏化；选择数值稀疏列再处理即可。

下面在 events 中加入普通文本列，捕获访问整张混合表时的特定异常。

In [8]:
mixed = events.assign(city=["北京", "上海", "广州", "深圳"])
try:
    mixed.sparse.density
except AttributeError as error:
    print(type(error).__name__)  # AttributeError，整表不是全部稀疏列。
else:
    raise AssertionError("预期混合 dtype 的整表 sparse 访问器不可用")

print(mixed["returns"].sparse.npoints)  # 1，单个稀疏列仍可使用访问器。
print(mixed[["returns", "complaints"]].sparse.density)  # 0.25。
print(mixed.shape)  # (4, 3)，普通列与稀疏列可以共存。

AttributeError
1
0.25
(4, 3)


## 6 稀疏计算与填充值变化

稀疏序列可以直接参与算术运算。运算不仅作用于显式值，也要作用于填充值，才能得到与普通数据一致的结果。

下面继续用开篇的 returns，给每个门店的件数加 1。结果中原来的零变为 1，相应填充值也变为 1；不能只给存储的 3 和 1 加一。

In [9]:
shifted = returns + 1

print(shifted.sparse.to_dense().tolist())  # [1, 1, 4, 1, 1, 2]。
print(shifted.sparse.fill_value)  # 1，填充值也参与加法。
print(shifted.sparse.npoints)  # 2，依然只需保存原来两个位置。
print(shifted.index.equals(returns.index))  # True。
assert shifted.sparse.to_dense().equals(dense_returns + 1)

[1, 1, 4, 1, 1, 2]
1
2
True


NumPy 通用函数也可作用于 SparseArray。下面以 abs 取绝对值，观察负填充值如何变成正填充值；结果仍是稀疏数组，不必先转为 NumPy 稠密数组。

In [10]:
signed = pd.arrays.SparseArray([-1, -1, -3, -1, 2], dtype="int64", fill_value=-1)
absolute = np.abs(signed)

print(type(absolute).__name__)  # SparseArray。
print(absolute.to_dense())  # [1 1 3 1 2]。
print(absolute.fill_value)  # 1，abs(-1) 的结果。
print(absolute.dtype)  # Sparse[int64, 1]。

SparseArray
[1 1 3 1 2]
1
Sparse[int64, 1]


不要把 npoints 或 density 当作数学上的非零数量或比例。填充值可能非零，而且部分运算会保留已经存储的位置，即使对应结果又变成填充值。

下面把 returns 乘零。当前实现保留原来两个显式位置，因此存储条目数不自动归零；这是存储表示的差异，完整数据仍全部为零。

In [11]:
all_zero = returns * 0

print(all_zero.sparse.to_dense().tolist())  # [0, 0, 0, 0, 0, 0]。
print(all_zero.sparse.sp_values)  # [0 0]，原来的两个存储位置还在。
print(all_zero.sparse.npoints, all_zero.sparse.density)  # 2，约 0.3333。
print(all_zero.sum())  # 0，存储条目数不是业务统计值。

[0, 0, 0, 0, 0, 0]
[0 0]
2 0.3333333333333333
0


两个稀疏 Series 相加时，仍遵循 pandas 的标签对齐。稀疏表示不会把 Series 自动变成纯位置数组；下面把右侧标签打乱，再逐标签核对结果。

In [12]:
left = pd.Series([0, 2, 0], index=["A", "B", "C"], dtype=pd.SparseDtype("int64", 0))
right = pd.Series([3, 0, 1], index=["C", "A", "B"], dtype=pd.SparseDtype("int64", 0))
combined = left + right

print(combined)  # A 为 0+0=0，B 为 2+1=3，C 为 0+3=3。
print(combined.index.tolist(), combined.shape)  # ['A', 'B', 'C'] (3,)。
print(combined.dtype)  # Sparse[int64, 0]。
expected = left.sparse.to_dense() + right.sparse.to_dense()
assert combined.sparse.to_dense().equals(expected)

A    0
B    3
C    3
dtype: Sparse[int64, 0]
['A', 'B', 'C'] (3,)
Sparse[int64, 0]


## 7 转换为稠密数据

sparse.to_dense 返回完整的普通 Series 或 DataFrame，保留 pandas 标签。Series.to_numpy 或对 SparseArray 调用 np.asarray 则得到普通 ndarray，包含全部位置，不带行标签。

稠密化需要为完整形状准备数据空间；to_numpy 的 copy=False 也不保证不分配内存。不要因原稀疏对象很小，就假定转换后的数组同样小。下面只对六个元素的 returns 做转换。

In [13]:
dense_back = returns.sparse.to_dense()
numpy_values = returns.to_numpy()
numpy_from_array = np.asarray(returns.array)

print(dense_back.equals(dense_returns))  # True，值、标签与 dtype 一致；equals 不检查 Series 名称。
print(dense_back.name == dense_returns.name)  # True，名称另行比较。
print(numpy_values)  # [0 0 3 0 0 1]，不只是两个显式存储值。
print(numpy_values.shape, numpy_values.dtype)  # (6,) int64。
print(np.array_equal(numpy_values, numpy_from_array))  # True。
print(returns.sparse.sp_values.shape)  # (2,)，只表示存储的值，不能当作完整数据。
print(returns.memory_usage(index=False), numpy_values.nbytes)  # 本机为 24、48 字节。

True
True
[0 0 3 0 0 1]
(6,) int64
True
(2,)
24 48


## 8 密度与内存估计

稀疏表示除了保存数值，还需要保存位置。显式值越来越多时，这部分开销可能抵消节省，甚至超过普通数组。

下面固定长度为 20、数值类型为 int64，改变非零数量。memory_usage(index=False, deep=True) 排除行索引，仅比较 Series 的数据存储估计；这不是整个 Python 进程的内存或转换过程的峰值。各次都先核对完整数据相同。

In [14]:
rows = []
for count in [0, 2, 10, 20]:
    values = np.zeros(20, dtype="int64")
    values[:count] = 1
    dense = pd.Series(values)
    sparse = dense.astype(pd.SparseDtype("int64", 0))
    assert sparse.sparse.to_dense().equals(dense)
    rows.append(
        {
            "nonzero": count,
            "density": sparse.sparse.density,
            "dense_bytes": dense.memory_usage(index=False, deep=True),
            "sparse_bytes": sparse.memory_usage(index=False, deep=True),
        }
    )

print(pd.DataFrame(rows))
# 本机密度为 0、0.1、0.5、1.0；稠密数据均为 160 字节。
# 对应稀疏数据为 0、24、120、240 字节，满密度时更大。
# 0 字节仅指这一口径下的底层数据，不表示整个 Series 对象不占内存。

   nonzero  density  dense_bytes  sparse_bytes
0        0      0.0          160             0
1        2      0.1          160            24
2       10      0.5          160           120
3       20      1.0          160           240


相同密度下，数值类型和填充值也会影响结果。较小的数值 dtype 使普通数组更紧凑，而稀疏位置记录仍有成本；填充值选得不合适则会存下更多条目。

下面只使用能由 int8 安全表示的 0、1，并继续比较相同四个整数在填充值为 0 或 1 时的表示。不把这些小样本数字推广为所有 dtype 的固定收益阈值。

In [15]:
small_values = pd.Series([0] * 10 + [1] * 10, dtype="int8")
small_sparse = small_values.astype(pd.SparseDtype("int8", 0))
print(small_sparse.sparse.density)  # 0.5。
print(small_values.memory_usage(index=False), small_sparse.memory_usage(index=False))
# 本机为 20、50 字节；此时稠密 int8 更小。

values = [1, 1, 1, 2]
fill_zero = pd.Series(values, dtype=pd.SparseDtype("int64", 0))
fill_one = pd.Series(values, dtype=pd.SparseDtype("int64", 1))
print(fill_zero.memory_usage(index=False), fill_one.memory_usage(index=False))
# 本机为 48、12 字节，填充值为 1 时只存一个条目。
assert fill_zero.sparse.to_dense().equals(fill_one.sparse.to_dense())

0.5
20 50
48 12


## 9 选学：SciPy 互操作入口
需要把二维数值交给支持 SciPy 稀疏矩阵的算法时，可以考虑下列入口。SciPy 需要额外安装，本章不执行这部分转换。

| API | 中文名称／含义 | 使用前检查 |
| --- | --- | --- |
| DataFrame.sparse.from_spmatrix | 从 SciPy 稀疏矩阵创建稀疏表 | 输入需可转换为 CSC 格式，明确提供行列标签 |
| DataFrame.sparse.to_coo | 将稀疏表转换为 COO 矩阵 | 检查数值 dtype，单独保存行列标签与顺序 |

矩阵里的行列是位置，不会自动带上 pandas 的业务标签。多个不同数值 dtype 可能合并成共同类型，转换后应检查 dtype、形状和数值。

SciPy 稀疏表示以零作为未存储位置的值；转换前应确认 pandas 各列的填充值为零，不能把非零或缺失填充值直接当作隐式零。若要调整表示，先明确逻辑数据应如何保留，再比较转换前后的结果；不通过改一个填充值属性来代替数据转换。

## 本章小结

（1）SparseArray 保存显式值与位置，SparseDtype 描述数值类型和填充值。稀疏存储可以省略重复值，但不删除对应记录。

（2）填充值与缺失含义不同；0 可以是有效观测，NaN 可以是填充值或显式存储值。统计应使用逻辑数据，不能用 npoints 代替有效样本数。

（3）运算会改变显式值，也可能改变填充值；标签仍然对齐。运算后的存储条目数不一定等于数学非零数。

（4）稠密化按完整形状展开。稀疏是否省内存取决于密度、dtype、填充值和位置表示，内存估计与进程峰值不是同一口径。

## 练习

（1）将下面的访问次数转成以 0 为填充值的稀疏 Series，保留标签和名称，核对总量、npoints 和 density。随后数据改成“大多数值为 1”，重新选择填充值并解释理由；选择不得改变原始逻辑值。

In [16]:
visits = pd.Series([0, 0, 5, 0, 0], index=["A", "B", "C", "D", "E"], name="visits")
changed = pd.Series([1, 1, 5, 1, 1], index=visits.index, name="visits")
# 在此分别选择 SparseDtype，打印存储条目数，并稠密化后比较原输入。
# 检查：第一组总量为 5，只需一个显式条目；两组都保留五条标签。

（2）预测下面相加和乘零后的完整值、填充值与存储条目数，再运行核对。解释为什么不能从“全部结果为零”直接推断 npoints 为零。

In [17]:
exercise = pd.Series([0, 2, 0, 3], dtype=pd.SparseDtype("int64", 0))
shifted = exercise + 2
cleared = exercise * 0
print(shifted.sparse.to_dense())
print(shifted.sparse.fill_value, shifted.sparse.npoints)
print(cleared.sparse.to_dense())
print(cleared.sparse.fill_value, cleared.sparse.npoints)
# 运行后分别核对逻辑数据与存储条目，不把二者混为一谈。

0    2
1    4
2    2
3    5
dtype: int64
2 2
0    0
1    0
2    0
3    0
dtype: int64
0 2


（3）同一输入分别用 0 和 NaN 作为稀疏填充值，比较缺失位置、count、sum、density。新增约束为“缺失代表尚未上报，不能视作零”，说明是否能调用 fillna(0)，以及为什么存储填充值仍然可以选为 0。

In [18]:
values = [0.0, np.nan, 0.0, 6.0, np.nan]
# 在此构造两种 SparseDtype，比较统计与稠密化结果。
# 检查：两种表示都保留两个缺失、三个有效观测，总量为 6。

（4）下面两个长度相同的 int64 序列是否都适合稀疏存储？比较不包含行索引的 memory_usage，并检查转换前后的值相同。若接收接口只接受 NumPy ndarray，再解释原有稀疏优势能否保持，以及需要检查哪种内存口径。

In [19]:
mostly_zero = pd.Series([0] * 18 + [2, 3], dtype="int64")
mostly_nonzero = pd.Series([1] * 18 + [2, 3], dtype="int64")
# 在此分别构造零填充值的稀疏版本，核对完整值并比较内存估计。
# 再仅对这些小输入调用 to_numpy，检查 shape、dtype、nbytes。
# 说明：不要用原稀疏数据的存储字节数代替稠密输出所需空间。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Sparse data structures](https://pandas.pydata.org/docs/user_guide/sparse.html) 的 SparseArray、SparseDtype、Sparse accessor、Sparse calculation、Conversion 与 Interaction with scipy.sparse；[SparseArray](https://pandas.pydata.org/docs/reference/api/pandas.arrays.SparseArray.html) 的 fill_value、dtype、kind 位置存储；[SparseDtype](https://pandas.pydata.org/docs/reference/api/pandas.SparseDtype.html) 的默认填充值；Series.sparse 的 [density](https://pandas.pydata.org/docs/reference/api/pandas.Series.sparse.density.html)、[npoints](https://pandas.pydata.org/docs/reference/api/pandas.Series.sparse.npoints.html)、[sp_values](https://pandas.pydata.org/docs/reference/api/pandas.Series.sparse.sp_values.html)；[DataFrame.sparse.to_dense](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sparse.to_dense.html) 的值与形状恢复；[Series.isna](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html)、[count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html)、[sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html)、[mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) 的缺失统计口径；[Series.equals](https://pandas.pydata.org/docs/reference/api/pandas.Series.equals.html) 的值与标签比较（名称边界另核对本机 3.0.6 官方 pandas/core/generic.py 的 NDFrame.equals）；[Series.add](https://pandas.pydata.org/docs/reference/api/pandas.Series.add.html) 的标签算术；[Series.memory_usage](https://pandas.pydata.org/docs/reference/api/pandas.Series.memory_usage.html) 的 index、deep 和字节口径；[Series.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_numpy.html) 的扩展类型转换及 copy 条件；[from_spmatrix](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sparse.from_spmatrix.html)、[to_coo](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sparse.to_coo.html) 的互操作入口、标签与共同类型。 |
| pandas 官方源码（raw.githubusercontent.com，v3.0.5；同时对照本机 3.0.6） | [sparse/array.py](https://raw.githubusercontent.com/pandas-dev/pandas/v3.0.5/pandas/core/arrays/sparse/array.py) 的 density、npoints、nbytes、fillna、&#95;arith&#95;method、&#95;wrap&#95;result：存储条目统计、数值与位置字节数、标量运算后保留显式位置；[sparse/accessor.py](https://raw.githubusercontent.com/pandas-dev/pandas/v3.0.5/pandas/core/arrays/sparse/accessor.py) 的 SparseAccessor.to_dense、SparseFrameAccessor.&#95;validate 与 to_coo：Series 标签保留、整表稀疏类型条件及互操作数据提取。 |
| NumPy 官方文档（NumPy 2.5） | [asarray](https://numpy.org/doc/2.5/reference/generated/numpy.asarray.html) 的 ndarray 转换；[absolute](https://numpy.org/doc/2.5/reference/generated/numpy.absolute.html) 的逐元素绝对值；[ndarray.nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的元素占用字节及不包含对象其他属性的边界。 |
| SciPy 官方文档（1.18，仅选学入口） | [Sparse arrays](https://docs.scipy.org/doc/scipy/reference/sparse.html) 的 COO、CSC、CSR 等表示及数组／矩阵接口；[Sparse Arrays 教程](https://docs.scipy.org/doc/scipy/tutorial/sparse.html) 的 Sparse arrays, implicit zeros, and duplicates：隐式零与显式存储条目。与 pandas 的具体转换条件结合上述 pandas API 和源码核对。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[sparse](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/sparse.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |